# 1. Import libraries and Config

In [25]:
import torch
import torch.nn as nn
import torchvision
from torchinfo import summary


from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
import importlib

# my Modules 
from Modularity.going_modular import data_setup, model_builder, utils, evaluation

### Color configure

In [6]:
RED = "\033[31m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
MAGENTA = "\033[35m"
CYAN = "\033[36m"
ORANGE = "\033[38;5;208m"
BOLD = "\033[1m"
RESET = "\033[0m"

In [7]:
# Device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"I'll be using {BOLD}{ORANGE}{device} {RESET}device")

I'll be using cpu device


# 2. Pre-Processing
### 2.1 Get the Data

In [8]:
# Data directory
DATA_DIR = ROOT_DIR / "Dataset/food_images/pizza_steak_sushi"
train_data_dir = DATA_DIR / "train"
test_data_dir = DATA_DIR / "test"

### 2.2. Turn to DataLoader

In [9]:
# We can create transform using manually or automaticcally. wehen manually it is more customze and best but there is one problem the pretrained model will not much. and the authomatic is fine with pretrained model but it is rigid there is not change to customize.

#  For now i use automatic transformation

weight = torchvision.models.EfficientNet_B0_Weights.DEFAULT
transforms = weight.transforms()

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
                                                train_dir=train_data_dir,
                                                test_dir=test_data_dir,
                                                train_transform=transforms,
                                                test_transform=transforms,
                                                batch_size=32,
                                                num_workers=0
                                            )
display(transforms)
train_dataloader, test_dataloader, class_names

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

(<torch.utils.data.dataloader.DataLoader at 0x715a93163d40>,
 ['pizza', 'steak', 'sushi'])

# 3. Get Pre-Trained and Train Model
### 3.1 Get Pre-Trained Model

In [10]:
model = torchvision.models.efficientnet_b0(weights=weight).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [29]:
# Then freeze the model parameters extractor
for param in model.features.parameters():
    param.requires_grad = False

# seed for reproducibility
import src.helper as helper
helper = importlib.reload(helper)

helper.set_seed(42)
# Update the classifier head
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280, 
                    out_features=len(class_names), bias=True)
).to(device)

In [12]:
summary(model, 
        input_size=(32, 3, 224, 224), # make sure this is "input_size", not "input_shape" (batch_size, color_channels, height, width)
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

### 3.2 Train and Track Pre-Train Model

In [13]:
# Define loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
try:
    from torch.utils.tensorboard import SummaryWriter
except:
    print("[INFO] Couldn't find tensorboard... installing it.")
    %pip install -q tensorboard
    from torch.utils.tensorboard import SummaryWriter

In [15]:
writter = SummaryWriter()
from src import engine


# Train the model with new Updated script
result = engine.train_model(
                    model=model,
                    train_dataloader=train_dataloader,
                    test_dataloader=test_dataloader,
                    optimizer=optimizer,
                    loss_fn=loss_fn,
                    epochs=5,
                    device=device,    
                    writer=writter) 

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0294 | train_accu: 0.4219 | test_loss: 0.8288 | test_accu: 0.8153


 20%|██        | 1/5 [00:32<02:10, 32.52s/it]

Epoch: 2 | train_loss: 0.8542 | train_accu: 0.6172 | test_loss: 0.6980 | test_accu: 0.9167


 40%|████      | 2/5 [01:03<01:35, 31.69s/it]

Epoch: 3 | train_loss: 0.7963 | train_accu: 0.6914 | test_loss: 0.6395 | test_accu: 0.9062


 60%|██████    | 3/5 [01:34<01:02, 31.18s/it]

Epoch: 4 | train_loss: 0.6802 | train_accu: 0.7695 | test_loss: 0.6663 | test_accu: 0.8759


 80%|████████  | 4/5 [02:05<00:31, 31.33s/it]

Epoch: 5 | train_loss: 0.5887 | train_accu: 0.9102 | test_loss: 0.5871 | test_accu: 0.8968


100%|██████████| 5/5 [02:38<00:00, 31.70s/it]


In [16]:
result

{'train_loss': [1.0293753817677498,
  0.8542311638593674,
  0.7963091805577278,
  0.6801904737949371,
  0.5886964835226536],
 'train_accu': [0.421875, 0.6171875, 0.69140625, 0.76953125, 0.91015625],
 'test_loss': [0.8287692268689474,
  0.6979528665542603,
  0.6394573052724203,
  0.6662943164507548,
  0.5871158639589945],
 'test_accu': [0.8153409090909092,
  0.9166666666666666,
  0.90625,
  0.8759469696969697,
  0.8967803030303031]}

In [18]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 7214), started 0:01:10 ago. (Use '!kill 7214' to kill it.)

### 3.3 Create Helper Function

In [ ]:
# By creating helper function we can build SummetryWriter() instance.
"""Summerywiter() are a method which is used for log and track what is happning on the model while training the model. it is directly specified by log_dir() method."""
print()

In [33]:
writer = helper.create_writer(experiment_name="First_Experiment", 
                        model_name="efficientnet_b0", 
                        extra="Nigatie")

[INFO] Creating a new SummaryWriter with log_dir: logs/First_Experiment/efficientnet_b0/Nigatie/2026-03-31_21-27-38
